In [37]:
import h5py
import numpy as np
from tensorflow.keras import utils

with h5py.File("../data/Galaxy10_DECals.h5", "r") as f:
    images = f["images"][:].astype(np.float32)
    labels = f["ans"][:]

    ra = f["ra"][:]
    dec = f["dec"][:]
    redshift = f["redshift"][:]
    pxscale = f["pxscale"][:]


c:\Users\User\Desktop\Університет\4_курс\ДИПЛОМ\galaxy-classification\.galaxy_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [120]:
ra[0], dec[0]

(np.float64(331.66405522483547), np.float64(-0.4841546865670835))

In [121]:
df.head()

,objid,specobjid,u,g,r,i,z_mag,z,zErr,velDisp,...,petroR90_r,q_r,u_r,fracDeV_r,flags,plate,mjd,fiberID,ra,dec
0,1237663542609838270,420007424494168064,25,24,23,22,21,0,0,121,...,1,0,-0,0,105624285352208,373,51788,170,332,-0
1,1237660222063313094,421086870085068800,24,23,23,23,22,0,0,131,...,2,-0,0,0,246361773703440,374,51791,1,335,-1
2,1237660222066524290,425603120052070400,19,18,17,17,16,0,0,138,...,7,-0,0,1,35253360136792,378,52146,47,342,-1
3,1237660237098057905,425724341209032704,24,24,23,23,23,0,0,82,...,1,-0,0,1,387097114579216,378,52146,488,341,1
4,1237657191974568096,430245252608059392,25,23,24,23,23,0,0,132,...,0,1,-1,0,316728336843024,382,51816,551,350,1


In [38]:
len(ra)

17736

In [39]:
from astroquery.sdss import SDSS
from astropy import coordinates as coords
import astropy.units as u

In [40]:
query = """
SELECT TOP 1 name
FROM sys.columns
WHERE object_id = OBJECT_ID('PhotoObj')
"""

result = SDSS.query_sql(query)
print([row['name'] for row in result])


[np.str_('objID')]


In [41]:
query = "SELECT TOP 1 * FROM PhotoObj"
result = SDSS.query_sql(query)

print(result)


                                        <html><head>                                       
-------------------------------------------------------------------------------------------
                                                             <title>Skyserver Error</title>
                                                                </head><body bgcolor=white>
<p>An error occured</p><p><font color=red><br> <html><body><h1>503 Service Unavailable</h1>
                                             No server is available to handle this request.
                                                                             </body></html>
                                                                  </font></p></BODY></HTML>


In [42]:
import requests
import pandas as pd

SDSS_URL = "https://skyserver.sdss.org/dr19/SkyServerWS/SearchTools/SqlSearch"


def run_sdss_sql(query):
    params = {
        "cmd": query,
        "format": "json"
    }

    try:
        response = requests.get(SDSS_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        # SDSS returns: {"Rows": [...], "Columns": [...]}
        rows = data[0]["Rows"] if isinstance(data, list) else data["Rows"]
        columns = data[0].get("Columns", None) if isinstance(data, list) else None

        if not rows:
            return pd.DataFrame()

        df = pd.DataFrame(rows)

        # optional: set column names if provided
        if columns and len(columns) == len(df.columns):
            df.columns = columns

        return df

    except Exception as e:
        print("SDSS error:", e)
        return None


In [56]:
import requests 
url = "https://skyserver.sdss.org/dr19/SkyServerWS/SearchTools/SqlSearch" 
query =  """
SELECT TOP 1
    p.objid,
    s.specobjid,
    p.ra,
    p.dec,

    p.u,
    p.g,
    p.r,
    p.i,
    p.z AS z_mag,

    s.z,
    s.zErr,
    s.velDisp,
    s.snMedian,
    s.class,
    s.subClass,

    p.type,
    p.petroR50_r,
    p.petroR90_r,
    p.q_r,
    p.u_r,
    p.fracDeV_r,
    p.flags,

    s.plate,
    s.mjd,
    s.fiberID

FROM PhotoObjAll AS p
LEFT JOIN SpecObj AS s
    ON p.objid = s.bestobjid

"""
params = { "cmd": query, "format": "json" } 
response = requests.get(url, params=params) 
data = response.json() 

In [57]:
data

[{'TableName': 'Table1',
  'Rows': [{'objid': 1237645876861272065,
    'specobjid': None,
    'ra': 336.438801475351,
    'dec': -0.834257465037838,
    'u': 11.07133,
    'g': 9.681775,
    'r': 12.46537,
    'i': 9.3345,
    'z_mag': 12.2708,
    'z': None,
    'zErr': None,
    'velDisp': None,
    'snMedian': None,
    'class': '',
    'subClass': '',
    'type': 3,
    'petroR50_r': 3.073734,
    'petroR90_r': 7.682547,
    'q_r': 0.1475463,
    'u_r': -0.01744305,
    'fracDeV_r': 0.5344208,
    'flags': 488262955569166,
    'plate': None,
    'mjd': None,
    'fiberID': None}]},
 {'TableName': 'SqlQuery',
  'Rows': [{'query': '\nSELECT TOP 1\n p.objid,\n s.specobjid,\n p.ra,\n p.dec,\n\n p.u,\n p.g,\n p.r,\n p.i,\n p.z AS z_mag,\n\n s.z,\n s.zErr,\n s.velDisp,\n s.snMedian,\n s.class,\n s.subClass,\n\n p.type,\n p.petroR50_r,\n p.petroR90_r,\n p.q_r,\n p.u_r,\n p.fracDeV_r,\n p.flags,\n\n s.plate,\n s.mjd,\n s.fiberID\n\nFROM PhotoObjAll AS p\nLEFT JOIN SpecObj AS s\n ON p.objid

In [115]:
pdf_rows = []

columns_to_gather = [
    "objid", "specobjid",
    "u", "g", "r", "i", "z_mag",
    "z", "zErr", "velDisp", "snMedian",
    "class", "subClass",
    "type", "petroR50_r", "petroR90_r",
    "q_r", "u_r", "fracDeV_r",
    "flags", "plate", "mjd", "fiberID"
]


def gather_metadata(ra, dec):
    delta = 10 / 3600

    query = f"""
    SELECT TOP 1
        p.objid,
        s.specobjid,
        p.ra,
        p.dec,

        p.u, p.g, p.r, p.i,
        p.z AS z_mag,

        s.z,
        s.zErr,
        s.velDisp,
        s.snMedian,
        s.class,
        s.subClass,

        p.type,
        p.petroR50_r,
        p.petroR90_r,
        p.q_r,
        p.u_r,
        p.fracDeV_r,
        p.flags,

        s.plate,
        s.mjd,
        s.fiberID

    FROM PhotoObj AS p
    OUTER APPLY (
        SELECT TOP 1 *
        FROM SpecObj AS s
        WHERE s.ra BETWEEN p.ra - 0.003 AND p.ra + 0.003
        AND s.dec BETWEEN p.dec - 0.003 AND p.dec + 0.003
        ORDER BY POWER(s.ra - p.ra, 2) + POWER(s.dec - p.dec, 2)
    ) s

    WHERE
        p.ra BETWEEN {ra - delta} AND {ra + delta}
        AND p.dec BETWEEN {dec - delta} AND {dec + delta}
    """



    df = run_sdss_sql(query)

    if df is None or df.empty:
        return None

    row = df.iloc[0]

    metadata = {col: None for col in columns_to_gather}

    for col in columns_to_gather:
        if col in df.columns:
            metadata[col] = row[col]

    metadata["ra"] = ra
    metadata["dec"] = dec

    pdf_rows.append(metadata)

    return metadata


In [116]:
for i in range(len(ra)):
    print(f"Gathering metadata for object {i+1}/{len(ra)}: RA={ra[i]}, Dec={dec[i]}")
    gather_metadata(ra[i], dec[i])

Gathering metadata for object 1/17736: RA=331.66405522483547, Dec=-0.4841546865670835
Gathering metadata for object 2/17736: RA=334.53657795329127, Dec=-1.1890306795735184
Gathering metadata for object 3/17736: RA=341.9024899263222, Dec=-1.1274181175484184
Gathering metadata for object 4/17736: RA=341.3433051172792, Dec=0.6581762089777219
Gathering metadata for object 5/17736: RA=349.1899823087505, Dec=0.9265433732484696
SDSS error: 403 Client Error: Forbidden for url: https://skyserver.sdss.org/dr19/SkyServerWS/SearchTools/SqlSearch?cmd=%0A++++SELECT+TOP+1%0A++++++++p.objid%2C%0A++++++++s.specobjid%2C%0A++++++++p.ra%2C%0A++++++++p.dec%2C%0A%0A++++++++p.u%2C+p.g%2C+p.r%2C+p.i%2C%0A++++++++p.z+AS+z_mag%2C%0A%0A++++++++s.z%2C%0A++++++++s.zErr%2C%0A++++++++s.velDisp%2C%0A++++++++s.snMedian%2C%0A++++++++s.class%2C%0A++++++++s.subClass%2C%0A%0A++++++++p.type%2C%0A++++++++p.petroR50_r%2C%0A++++++++p.petroR90_r%2C%0A++++++++p.q_r%2C%0A++++++++p.u_r%2C%0A++++++++p.fracDeV_r%2C%0A++++++++p.flag

In [117]:
df = pd.DataFrame(pdf_rows)

In [118]:
pdf_rows

[{'objid': np.int64(1237663542609838270),
  'specobjid': np.int64(420007424494168064),
  'u': np.float64(25.36423),
  'g': np.float64(23.84239),
  'r': np.float64(22.71444),
  'i': np.float64(21.81713),
  'z_mag': np.float64(21.26674),
  'z': np.float64(0.08191887),
  'zErr': np.float64(1.15505e-05),
  'velDisp': np.float64(121.0042),
  'snMedian': np.float64(10.16239),
  'class': 'GALAXY',
  'subClass': 'STARFORMING',
  'type': np.int64(3),
  'petroR50_r': np.float64(0.4926183),
  'petroR90_r': np.float64(1.062742),
  'q_r': np.float64(0.09368636),
  'u_r': np.float64(-0.08591975),
  'fracDeV_r': np.int64(0),
  'flags': np.int64(105624285352208),
  'plate': np.int64(373),
  'mjd': np.int64(51788),
  'fiberID': np.int64(170),
  'ra': np.float64(331.66405522483547),
  'dec': np.float64(-0.4841546865670835)},
 {'objid': np.int64(1237660222063313094),
  'specobjid': np.int64(421086870085068800),
  'u': np.float64(24.25105),
  'g': np.float64(22.82115),
  'r': np.float64(22.7092),
  'i': n

In [ ]:
pdf_rows.sav

In [119]:
df.to_csv("../data/metadata_new.csv", index=False)

In [109]:
import seaborn as sns

In [114]:
df

,objid,specobjid,ra,dec,u,g,r,i,z_mag,z,...,type,petroR50_r,petroR90_r,q_r,u_r,fracDeV_r,flags,plate,mjd,fiberID
0,1237663542609838270,420007424494168064,332,-0,25,24,23,22,21,0,...,3,0,1,0,-0,0,105624285352208,373,51788,170
1,1237660222063313094,421086870085068800,335,-1,24,23,23,23,22,0,...,3,1,2,-0,0,0,246361773703440,374,51791,1
2,1237660222066524290,425603120052070400,342,-1,19,18,17,17,16,0,...,3,2,7,-0,0,1,35253360136792,378,52146,47
3,1237660237098057905,425724341209032704,341,1,24,24,23,23,23,0,...,6,0,1,-0,0,1,387097114579216,378,52146,488
4,1237663278459912331,430205670189459456,349,1,24,24,23,22,22,0,...,6,1,1,-0,-0,0,387099262063376,382,51816,407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17273,1237667783913701534,2984862158602397696,198,23,19,17,16,16,16,0,...,3,3,8,0,-0,0,35191083110416,2651,54507,369
17274,1237667324338503817,2528871034648553472,205,27,18,17,16,16,16,0,...,3,3,8,0,0,0,52776826703888,2246,53767,363
17275,1237662238542135486,1388375093616338944,190,11,19,23,23,23,22,0,...,3,11,13,3,-0,1,316730484330768,1233,52734,511
17276,1237664817672421511,2376929792650930176,171,36,19,17,17,16,16,0,...,3,3,8,0,-0,1,35253360136208,2111,53467,564


In [83]:
nulls = 0
for i in range(1):
    
    ra_search = ra[i]
    dec_search = dec[i]

    pos = coords.SkyCoord(ra=ra_search*u.deg, dec=dec_search*u.deg)

    spec_result = SDSS.query_region(
        pos, radius=10*u.arcsec,
        specobj_fields=["specobjid", "z", "velDisp", "class", "subClass", "plate", "mjd", "fiberID"]
    )
    
    if spec_result is not None and len(spec_result) > 0:
        row = spec_result[0]
        print(f"Spec result found for position {i}: {row['specobjid']}")
    else:
        nulls += 1
        print(f"No spec result found for position {i}")


Spec result found for position 0: 420007424494168064


In [103]:

ra_search = 194.5939288766277
dec_search = 2.7939167633484434

pos = coords.SkyCoord(ra=ra_search*u.deg, dec=dec_search*u.deg)

phot_result = SDSS.query_region(
    pos, radius=10*u.arcsec,
    photoobj_fields=["objid", "ra", "dec", "u", "g", "r", "i", "z", "type", "petroR50_r", "petroR90_r", "flags"]
)

spec_result = SDSS.query_region(
    pos, radius=10*u.arcsec,
    specobj_fields=["specobjid", "z", "velDisp", "class", "subClass", "plate", "mjd", "fiberID"]
)

metadata = {}

if phot_result is not None and len(phot_result) > 0:
    row = phot_result[0] 
    metadata.update({
        "phot_objid": row["objid"],
        "phot_ra": row["ra"],
        "phot_dec": row["dec"],
        "phot_type": row["type"],
        "u_mag": row["u"],
        "g_mag": row["g"],
        "r_mag": row["r"],
        "i_mag": row["i"],
        "z_mag": row["z"],
        "petroR50_r": row.get("petroR50_r"),
        "petroR90_r": row.get("petroR90_r"),
        "flags": row.get("flags")
    })

if spec_result is not None and len(spec_result) > 0:
    row = spec_result[0]  # take the closest match
    metadata.update({
        "specobjid": row["specobjid"],
        "z": row["z"],
        "velDisp": row.get("velDisp"),
        "class": row.get("class"),
        "subClass": row.get("subClass"),
        "plate": row.get("plate"),
        "mjd": row.get("mjd"),
        "fiberID": row.get("fiberID")
    })

print("--- Full SDSS Metadata ---")
for key, value in metadata.items():
    print(f"{key}: {value}")


--- Full SDSS Metadata ---
phot_objid: 1237651736837095466
phot_ra: 194.593015921478
phot_dec: 2.79264788527557
phot_type: 3
u_mag: 16.53813
g_mag: 15.23812
r_mag: 14.8281
i_mag: 14.6183
z_mag: 14.50701
petroR50_r: 13.60397
petroR90_r: 31.89257
flags: 35287719874576


In [123]:
phot_result

objid,ra,dec,u,g,r,i,z,type,petroR50_r,petroR90_r,flags
uint64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,int64
1237651736837095466,194.593015921478,2.79264788527557,16.53813,15.23812,14.8281,14.6183,14.50701,3,13.60397,31.89257,35287719874576
1237651736837095468,194.594592290429,2.79343298414436,21.47583,21.66706,22.36824,24.3618,22.82689,6,0.649678,1.147116,545426788978960


In [11]:
query = "SELECT TOP 10 * FROM PhotoObjAll WHERE objid = {}".format(metadata.get("phot_objid", 50))
all_fields = SDSS.query_sql(query)
print(all_fields)
all_fields.show_in_browser()

       objID        skyVersion run  ...      TAI_i            TAI_z      
------------------- ---------- ---- ... ---------------- ----------------
1237651736837095466          2 1458 ... 4464142122.79734 4464142266.19318


In [31]:
import matplotlib.pyplot as plt
plt.imshow(images[50]/255.0)


In [11]:
images[0].shape

(256, 256, 3)

In [ ]:
import cv2

idx = 0
img = images[455]

# convert to uint8 if needed
if img.dtype != np.uint8:
    img = (img - img.min()) / (img.max() - img.min()) * 255
    img = img.astype(np.uint8)


# RGB → BGR for OpenCV
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

cv2.imshow("Galaxy", img_bgr)
cv2.waitKey(0)
cv2.destroyAllWindows()
